# C2 · Text-to-image retrieval on Flickr30k (12 min)

**Question:** can one embedding space find the right photo from a sentence, and the right sentence from a photo?

Flow: embed 1,000 images and 5,000 captions → Recall@K in both directions → your own queries → failure analysis → optional bigger model.
Outputs go to `outputs/c2/`.

In [ ]:
# Make the project folder importable (works from notebooks/ or the project root)
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# ---- configuration -------------------------------------------------------
CLIP_ID   = "openai/clip-vit-base-patch32"
N_IMAGES  = 1000                 # up to 1,000
QUERIES   = ["two dogs running on a beach", "a man riding a bicycle in the city",
             "children playing in the snow", "a crowd watching a concert at night"]
RUN_LARGE = False                # True: also evaluate clip-vit-large-patch14 (slower)

In [ ]:
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from PIL import Image
from vlmkit.common import get_device, out_dir, save_json, show_grid, Timer
from vlmkit.clip_utils import Clip, retrieval_recall
from vlmkit import data

DEVICE, OUT = get_device(), out_dir("c2")
items = data.load_flickr1k(limit=N_IMAGES)
images = [Image.open(it["path"]).convert("RGB") for it in items]
captions = [c for it in items for c in it["captions"]]
owner = np.array([i for i, it in enumerate(items) for _ in it["captions"]])   # caption -> image index
print(f"{len(images)} images, {len(captions)} captions, device={DEVICE}")

## 1. Embed images and captions (cached)

In [ ]:
def embed_all(clip_id):
    cache = OUT / f"emb_{clip_id.replace('/', '_')}_{len(images)}.npz"
    if cache.exists():
        z = np.load(cache); return z["V"], z["T"], float(z["sec"])
    clip = Clip(clip_id, DEVICE)
    with Timer() as t:
        V = clip.images(images, progress=True).numpy()
        T = clip.texts(captions).numpy()
    np.savez(cache, V=V, T=T, sec=t.seconds)
    return V, T, t.seconds

V, T, sec = embed_all(CLIP_ID)
print(f"embedding took {sec:.1f} s (cached afterwards)")

## 2. Recall@K in both directions

In [ ]:
def evaluate(V, T):
    S = T @ V.T                                                    # (captions, images)
    t2i = retrieval_recall(S, [{int(o)} for o in owner])
    i2t = retrieval_recall(S.T, [set(np.where(owner == i)[0]) for i in range(len(V))])
    return S, t2i, i2t

S, t2i, i2t = evaluate(V, T)
res = pd.DataFrame([{"direction": "text -> image", **t2i}, {"direction": "image -> text", **i2t}])
res.to_csv(OUT / "recall.csv", index=False)
res

In [ ]:
# Look inside the similarity matrix: 8 images x their first caption. A bright diagonal = retrieval works.
k = 8
first_cap = [int(np.where(owner == i)[0][0]) for i in range(k)]
fig = plt.figure(figsize=(10, 6.2))
gs = fig.add_gridspec(2, k, height_ratios=[1, 5], hspace=0.05)
for i in range(k):
    a = fig.add_subplot(gs[0, i]); a.imshow(images[i].resize((96, 96))); a.axis("off")
ax = fig.add_subplot(gs[1, :])
hm = ax.imshow(S[first_cap][:, :k], cmap="viridis")
ax.set_xticks(range(k)); ax.set_xticklabels([f"img {i}" for i in range(k)])
ax.set_yticks(range(k)); ax.set_yticklabels([captions[q][:55] for q in first_cap], fontsize=8)
for i in range(k):
    for j in range(k):
        ax.text(j, i, f"{S[first_cap[i], j]:.2f}", ha="center", va="center", fontsize=7, color="w")
fig.colorbar(hm, ax=ax, fraction=0.03, label="cosine similarity")
plt.savefig(OUT / "similarity_matrix.png", dpi=120, bbox_inches="tight"); plt.show()

## 3. Your own queries

In [ ]:
clip = Clip(CLIP_ID, DEVICE)
Q = clip.texts(QUERIES).numpy()
for q, qv in zip(QUERIES, Q):
    top = np.argsort(-(V @ qv))[:3]
    show_grid([images[i] for i in top], [f"{(V[i] @ qv):.3f}" for i in top], cols=3, size=2.6,
              save=OUT / f"query_{q[:30].replace(' ', '_')}.png")
    print("query:", q)

## 4. Failure analysis: which captions miss their image?

In [ ]:
rank = np.array([int((S[q] > S[q, owner[q]]).sum()) + 1 for q in range(len(captions))])   # 1 = best
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(np.clip(rank, 1, 51), bins=np.arange(1, 53) - 0.5, color="#4C78A8")
ax.set_xlabel("rank of the correct image (51 = 51 or worse)"); ax.set_ylabel("captions")
ax.set_title(f"{(rank == 1).mean():.0%} of captions rank their image first"); plt.tight_layout()
plt.savefig(OUT / "rank_histogram.png", dpi=120); plt.show()

worst = np.argsort(-rank)[:3]
for q in worst:
    top1 = int(np.argmax(S[q]))
    print(f"\ncaption: {captions[q]}\ncorrect image rank: {rank[q]}")
    show_grid([images[owner[q]], images[top1]], ["correct image", "model's top-1"], cols=2, size=2.8)
pd.DataFrame({"caption": captions, "rank": rank}).sort_values("rank", ascending=False).to_csv(
    OUT / "caption_ranks.csv", index=False)

## 5. Optional: a larger CLIP

In [ ]:
rows = [{"model": CLIP_ID, "embed_seconds": sec, **{f"t2i {k}": v for k, v in t2i.items()}}]
if RUN_LARGE:
    big = "openai/clip-vit-large-patch14"
    Vb, Tb, secb = embed_all(big)
    _, t2i_b, _ = evaluate(Vb, Tb)
    rows.append({"model": big, "embed_seconds": secb, **{f"t2i {k}": v for k, v in t2i_b.items()}})
cmp_df = pd.DataFrame(rows); cmp_df.to_csv(OUT / "model_comparison.csv", index=False)
cmp_df

In [ ]:
summary = {"clip": CLIP_ID, "images": len(images), "captions": len(captions),
           "text_to_image": t2i, "image_to_text": i2t,
           "median_rank_of_correct_image": float(np.median(rank)),
           "models": cmp_df.to_dict("records")}
save_json(summary, OUT / "summary.json")
summary

## 6. Try it live

**Sentence → photos:** type anything and press **Run**. **Photo → captions:** pick an image; ✓ marks the captions humans actually wrote for it.

In [ ]:
from vlmkit.common import live

def search_panel(query, top_k):
    s = V @ clip.texts([query]).numpy()[0]
    top = np.argsort(-s)[:top_k]
    show_grid([images[i] for i in top], [f"#{r + 1}   {s[i]:.3f}" for r, i in enumerate(top)],
              cols=min(top_k, 5), size=2.6)

def caption_panel(image_index):
    s = T @ V[image_index]
    show_grid([images[image_index]], [f"image {image_index}"], cols=1, size=3.5)
    for r, q in enumerate(np.argsort(-s)[:8]):
        print(f"{r + 1}. {'✓' if owner[q] == image_index else ' '} {s[q]:.3f}  {captions[q]}")

live(search_panel, query="a dog jumping to catch a frisbee", top_k=(1, 10, 1, 5))
live(caption_panel, image_index=(0, len(images) - 1, 1, 0))

## Exercises for students

1. Write five hard queries (counts, colours, relations such as "a dog to the left of a cat"). Which kinds of detail does CLIP miss?
2. Read the worst captions in `outputs/c2/caption_ranks.csv`: is the model wrong, or is the caption vague?
3. **Transfer to your project:** embed your video keyframes and search them with sentences ("white van stopped in the left lane").